# 05 · 训练流程：损失函数、优化器与训练循环

> 这一节把"模型、数据、损失、优化器、训练循环"串成完整流水线。

## 5.1 深度学习的标准训练流程

```
for epoch in range(num_epochs):
    # ===== 训练阶段 =====
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()      # 1
        pred = model(x)            # 2
        loss = criterion(pred, y)  # 3
        loss.backward()            # 4
        optimizer.step()           # 5
    # ===== 验证阶段 =====
    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            ...
```

## 5.2 损失函数

| 任务 | 损失函数 | 输入要求 |
|---|---|---|
| 回归 | `nn.MSELoss` | 预测值、真实值 (float) |
| 回归（抗异常值） | `nn.L1Loss` / `SmoothL1Loss` | float |
| 二分类 | `nn.BCEWithLogitsLoss` | logits（**不要先 sigmoid**）、float 标签 |
| 多分类 | `nn.CrossEntropyLoss` | logits `(N,C)`、long 标签 `(N,)` |
| 多标签 | `nn.BCEWithLogitsLoss` | logits + 多热标签 |
| 排序/对比 | `nn.TripletMarginLoss` | 嵌入向量 |

### 两个最大坑
1. `CrossEntropyLoss` 内部包含 `LogSoftmax + NLLLoss`，
   **模型最后一层不要加 softmax**，直接输出 logits。
2. `BCEWithLogitsLoss` 内部包含 sigmoid，**不要手动 sigmoid**。

In [ ]:
import torch
import torch.nn as nn

# 回归
pred = torch.tensor([[2.5], [0.0], [-1.0]])
target = torch.tensor([[2.0], [1.0], [-3.0]])
print("MSELoss :", nn.MSELoss()(pred, target).item())
print("L1Loss  :", nn.L1Loss()(pred, target).item())
print("SmoothL1:", nn.SmoothL1Loss()(pred, target).item())
# MSE = mean((0.5)^2 + (1)^2 + (2)^2) = (0.25+1+4)/3

In [ ]:
# 多分类：CrossEntropyLoss
logits = torch.tensor([[2.0, 1.0, 0.1],
                       [0.5, 2.5, 0.3]])     # 2 个样本，3 类
labels = torch.tensor([0, 2])                 # long 标签

ce = nn.CrossEntropyLoss()
loss = ce(logits, labels)
print("CrossEntropyLoss:", loss.item())

# 手动验证：softmax + nll
log_prob = torch.log_softmax(logits, dim=1)
manual = -log_prob[0, 0] + -log_prob[1, 2]
print("手动计算:", (manual / 2).item())

# 带类别权重（处理类别不平衡）
weights = torch.tensor([1.0, 5.0, 2.0])
ce_w = nn.CrossEntropyLoss(weight=weights)
print("带权 CE:", ce_w(logits, labels).item())

# 忽略某个类别（如 padding）
ce_ignore = nn.CrossEntropyLoss(ignore_index=-100)
print("ignore_index:", ce_ignore(logits, torch.tensor([0, -100])).item())

In [ ]:
# 二分类：BCEWithLogitsLoss
logits = torch.tensor([0.8, -1.2, 2.0])
targets = torch.tensor([1.0, 0.0, 1.0])   # float

bce = nn.BCEWithLogitsLoss()
print("BCEWithLogits:", bce(logits, targets).item())
print("sigmoid 后:", torch.sigmoid(logits))

# 对比：手动 sigmoid + BCELoss
bce2 = nn.BCELoss()
print("sigmoid+BCELoss:", bce2(torch.sigmoid(logits), targets).item())

In [ ]:
# 标签平滑（现代分类训练常用，防过自信）
loss_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)
logits = torch.randn(4, 5)
labels = torch.randint(0, 5, (4,))
print("label_smoothing CE:", loss_smooth(logits, labels).item())

## 5.3 优化器

| 优化器 | 特点 | 常用场景 |
|---|---|---|
| `SGD` | 最基础，需调 lr + momentum | 追求极致精度（CV） |
| `SGD(momentum=0.9)` | 带动量，收敛更快 | 经典 CNN |
| `Adam` | 自适应学习率，易收敛 | 通用默认 |
| `AdamW` | Adam + 正确权重衰减 | Transformer、现代首选 |
| `RMSprop` | 自适应 | RNN、GAN |
| `Adagrad` | 累积梯度平方 | 稀疏特征 |

创建优化器：`optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)`

In [ ]:
import torch.nn as nn

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))

# 不同优化器
sgd = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
adam = torch.optim.Adam(model.parameters(), lr=1e-3)
adamw = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
rms = torch.optim.RMSprop(model.parameters(), lr=1e-3)

for name, opt in [("SGD", sgd), ("Adam", adam), ("AdamW", adamw), ("RMSprop", rms)]:
    print(f"{name:8s} defaults: {opt.defaults}")

In [ ]:
# 分层学习率：给不同部分不同 lr（迁移学习常用）
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
opt = torch.optim.SGD([
    {"params": model[0].parameters(), "lr": 1e-4},   # 底层小 lr
    {"params": model[2].parameters(), "lr": 1e-3},   # 顶层大 lr
], lr=1e-3, momentum=0.9)

for g in opt.param_groups:
    print("lr =", g["lr"], " 参数数 =", sum(p.numel() for p in g["params"]))

### 权重衰减 weight_decay

- **L2 正则化**：在损失里加 $\lambda \|\theta\|^2$，防止过拟合。
- `weight_decay` 就是 $\lambda$。SGD 里等价于 L2；Adam 里 AdamW 才是正确的解耦实现。
- 常用值：`1e-4 ~ 5e-2`。**不要给 bias 和 norm 层加 weight_decay**。

In [ ]:
# 不给 bias / norm 加 weight_decay 的正确做法
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)
        self.bn = nn.BatchNorm1d(5)
        self.out = nn.Linear(5, 2)
    def forward(self, x):
        return self.out(self.bn(self.fc(x)))

net = Net()
decay, no_decay = [], []
for name, p in net.named_parameters():
    if p.ndim <= 1:          # bias、norm 的 weight
        no_decay.append(p)
    else:
        decay.append(p)

opt = torch.optim.AdamW([
    {"params": decay, "weight_decay": 0.01},
    {"params": no_decay, "weight_decay": 0.0},
], lr=1e-3)

print("decay 参数组:", sum(p.numel() for p in decay))
print("no_decay 参数组:", sum(p.numel() for p in no_decay))

## 5.4 学习率调度器（Scheduler）

学习率（lr）是最重要的超参数。常用策略：

| Scheduler | 说明 |
|---|---|
| `StepLR` | 每 N 个 epoch 乘 gamma |
| `MultiStepLR` | 在指定 epoch 衰减 |
| `ExponentialLR` | 每 epoch 指数衰减 |
| `CosineAnnealingLR` | 余弦退火，平滑 |
| `OneCycleLR` | 先升后降，超收敛 |
| `ReduceLROnPlateau` | 验证指标不降才降 lr |
| `LambdaLR` | 自定义函数（含 warmup） |

**注意**：大多数 scheduler 要在 `optimizer.step()` **之后**调用
（`ReduceLROnPlateau` 传验证 loss）。

In [ ]:
import matplotlib.pyplot as plt

def lr_curve(sched, steps):
    lrs = []
    for _ in range(steps):
        lrs.append(sched.get_last_lr()[0])
        sched.step()
    return lrs

m = nn.Linear(1, 1)

opt = torch.optim.SGD(m.parameters(), lr=0.1)
s1 = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

opt2 = torch.optim.SGD(m.parameters(), lr=0.1)
s2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=50)

opt3 = torch.optim.SGD(m.parameters(), lr=0.1)
s3 = torch.optim.lr_scheduler.ExponentialLR(opt3, gamma=0.9)

opt4 = torch.optim.SGD(m.parameters(), lr=0.1)
s4 = torch.optim.lr_scheduler.OneCycleLR(opt4, max_lr=0.5, total_steps=50)

plt.figure(figsize=(9, 4))
plt.plot(lr_curve(s1, 50), label="StepLR")
plt.plot(lr_curve(s2, 50), label="CosineAnnealing")
plt.plot(lr_curve(s3, 50), label="Exponential")
plt.plot(lr_curve(s4, 50), label="OneCycle")
plt.xlabel("epoch/step"); plt.ylabel("lr"); plt.legend(); plt.grid(alpha=0.3)
plt.title("常见学习率调度")
plt.show()

In [ ]:
# 带 warmup 的调度（LambdaLR）
def warmup_cosine(step):
    warmup = 5
    total = 50
    if step < warmup:
        return step / warmup
    progress = (step - warmup) / (total - warmup)
    import math
    return 0.5 * (1 + math.cos(math.pi * progress))

m = nn.Linear(1, 1)
opt = torch.optim.SGD(m.parameters(), lr=0.1)
sched = torch.optim.lr_scheduler.LambdaLR(opt, warmup_cosine)
print("warmup+cosine lr 序列:", [round(lr_curve(sched, 1)[0], 3) for _ in range(50)][:12], "...")

## 5.5 评价指标

| 任务 | 指标 |
|---|---|
| 分类 | 准确率 accuracy、精确率/召回率/F1、混淆矩阵 |
| 多分类 | Top-1 / Top-5 accuracy |
| 回归 | MSE / RMSE / MAE / R² |
| 分割 | IoU / Dice |

In [ ]:
def accuracy(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

logits = torch.randn(100, 5)
labels = torch.randint(0, 5, (100,))
print("accuracy:", accuracy(logits, labels))

# Top-k
def topk_accuracy(logits, labels, k=3):
    topk = logits.topk(k, dim=1).indices              # (N, k)
    correct = (topk == labels.unsqueeze(1)).any(dim=1)
    return correct.float().mean().item()

print("top-3 accuracy:", topk_accuracy(logits, labels, k=3))

## 5.6 完整训练模板（可直接复用）

下面写一个通用 `fit` 函数，包含：训练、验证、学习率调度、最佳模型保存、早停。

In [ ]:
import copy
from torch.utils.data import DataLoader, TensorDataset


def make_data(n=1000, dim=20, classes=3, seed=0):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, dim, generator=g)
    W = torch.randn(dim, classes, generator=g)
    y = (X @ W).argmax(dim=1)
    return TensorDataset(X, y)


class SimpleNet(nn.Module):
    def __init__(self, dim=20, classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, classes),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
import time


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n += y.size(0)
    return total_loss / total_n, total_correct / total_n


def fit(model, train_loader, val_loader, epochs=20, lr=1e-3,
        weight_decay=1e-4, device="cpu", patience=5, verbose=True):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc, best_state, best_epoch, bad = 0.0, None, -1, 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # ---- 训练 ----
        model.train()
        t0 = time.time()
        run_loss, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * y.size(0)
            n += y.size(0)
        train_loss = run_loss / n

        # ---- 验证 ----
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # ---- 保存最佳 & 早停 ----
        if val_acc > best_acc:
            best_acc, best_epoch, bad = val_acc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1

        if verbose and (epoch % 2 == 0 or epoch == 1):
            print(f"epoch {epoch:3d} | train_loss {train_loss:.4f} | "
                  f"val_loss {val_loss:.4f} | val_acc {val_acc:.4f} | "
                  f"lr {scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.1f}s")

        if bad >= patience:
            print(f"早停于 epoch {epoch}（最佳 epoch {best_epoch}，acc={best_acc:.4f}）")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_acc

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = make_data(2000, seed=0)
val_ds = make_data(500, seed=1)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

model = SimpleNet(dim=20, classes=3)
history, best_acc = fit(model, train_loader, val_loader, epochs=15, lr=3e-3, device=device)
print(f"\n最佳验证准确率: {best_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history["val_acc"], color="green")
axes[1].set_title("Val Accuracy"); axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5.7 梯度裁剪与梯度累加

- **梯度裁剪**：防止 RNN/Transformer 梯度爆炸。
- **梯度累加**：显存不够时的"虚拟大 batch"。

In [ ]:
# 梯度裁剪：两种方式
model = SimpleNet()
x = torch.randn(8, 20); y = torch.randint(0, 3, (8,))
loss = nn.CrossEntropyLoss()(model(x), y)
loss.backward()

# 方式 1：按范数裁剪（推荐）
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print("裁剪前总范数:", float(total_norm))

loss = nn.CrossEntropyLoss()(model(x), y)
loss.backward()
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=0.5)   # 方式 2：按值
print("裁剪完成")

In [ ]:
# 梯度累加：4 个小 batch ≈ 1 个 4 倍大 batch
model = SimpleNet()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
accum_steps = 4
loader = DataLoader(make_data(64), batch_size=16, shuffle=True)

optimizer.zero_grad()
for i, (x, y) in enumerate(loader):
    logits = model(x)
    loss = nn.CrossEntropyLoss()(logits, y) / accum_steps   # 关键：除以累加步数
    loss.backward()
    if (i + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
        print(f"在第 {i+1} 个 mini-batch 后更新了一次参数")

## 5.8 混合精度训练（AMP，GPU 提速）

```python
scaler = torch.cuda.amp.GradScaler()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(x)
    loss = criterion(logits, y)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```
可省 30%~50% 显存、提速 1.5~3x。详见第 12 节。

## 5.9 常见训练问题排查

| 现象 | 可能原因 | 对策 |
|---|---|---|
| loss = NaN | lr 太大 / log(0) / 除零 | 降 lr、加 eps、梯度裁剪 |
| loss 不下降 | lr 太小、数据/标签错、模型没激活 | 调 lr、检查数据、去掉最后 softmax |
| train 高 val 低 | 数据泄漏 / BN 问题 | 检查划分、勿在验证集 shuffle |
| train 低 val 高 | 过拟合 | 加正则、增强、早停、更多数据 |
| 准确率卡在类别比例 | 只学多数类 / lr 崩 | 类别权重、重采样、调 lr |
| 验证波动大 | batch 太小 / BN | 增大 batch、用 LayerNorm |

**调试神器**：拿 1 个 batch 反复训练，看能不能过拟合到 100%。

In [ ]:
# 过拟合单个 batch 测试（快速验证模型和训练流程是否正确）
model = SimpleNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
x = torch.randn(16, 20)
y = torch.randint(0, 3, (16,))

for step in range(200):
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(model(x), y)
    loss.backward()
    opt.step()
    if (step + 1) % 50 == 0:
        acc = (model(x).argmax(1) == y).float().mean().item()
        print(f"step {step+1:3d} | loss {loss.item():.4f} | acc {acc:.3f}")
print("\n若 acc 能到 1.0，说明模型/流程正常，问题在泛化。")

## 5.10 练习

**练习 1**：用 `SGD(momentum=0.9)` 和 `Adam` 分别训练 `SimpleNet`，比较收敛速度。

**练习 2**：用 `ReduceLROnPlateau` 替换 Cosine 调度，重跑 `fit`。

**练习 3**：把 `label_smoothing=0.1` 加到 `CrossEntropyLoss`，观察验证准确率变化。

**练习 4**：故意把 lr 设成 10，观察 loss 爆炸成 NaN。

**练习 5**：实现"3 个 epoch 内验证准确率不提升就停"的早停（`fit` 的 patience 已实现，改参数验证）。

In [ ]:
# 练习 1
for opt_name in ["SGD", "Adam"]:
    model = SimpleNet()
    if opt_name == "SGD":
        opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    tr = DataLoader(make_data(1000, seed=0), batch_size=64, shuffle=True)
    losses = []
    for epoch in range(10):
        tot = 0
        for x, y in tr:
            opt.zero_grad()
            loss = nn.CrossEntropyLoss()(model(x), y)
            loss.backward(); opt.step()
            tot += loss.item()
        losses.append(tot / len(tr))
    print(f"练习1 {opt_name:5s} 10 epoch 后 loss: {losses[-1]:.4f}")

# 练习 4：lr 过大
model = SimpleNet()
opt = torch.optim.SGD(model.parameters(), lr=10.0)
x = torch.randn(16, 20); y = torch.randint(0, 3, (16,))
for step in range(20):
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(model(x), y)
    loss.backward(); opt.step()
    if step % 5 == 0:
        print(f"\n练习4 step {step} loss = {loss.item()}")

## 5.11 小结

- 五步循环：`zero_grad → forward → loss → backward → step`。
- 损失：分类用 `CrossEntropyLoss`（输入 logits + long 标签）；二分类用 `BCEWithLogitsLoss`。
- 优化器：默认 `AdamW`；CV 追求精度可用 `SGD+momentum`。
- Scheduler 让 lr 随训练下降；`OneCycleLR`/warmup+cosine 很强。
- 训练必须 `model.train()`，推理必须 `model.eval() + no_grad()`。
- 显存不够：梯度累加 / AMP；不稳定：梯度裁剪 / 降 lr。

下一节：`06_实战MNIST手写数字识别.ipynb`